# 07 · Pérdida esperada y propuesta de puntos de corte

**Objetivo:** traducir el scorecard a dinero y a decisiones: cuánto se espera perder por banda de puntaje y a partir de qué puntaje conviene **aprobar**, **revisar** o **rechazar** una solicitud.

## Conceptos

### Pérdida esperada (EL, *Expected Loss*)

$$\text{EL} = \text{PD} \times \text{LGD} \times \text{EAD}$$

| Componente | Pregunta que responde |
|---|---|
| **PD** (*Probability of Default*) | ¿Qué probabilidad hay de que el cliente no pague? La da el scorecard. |
| **LGD** (*Loss Given Default*) | Si no paga, ¿qué % de lo que debe se pierde, después de cobranza y recuperaciones? |
| **EAD** (*Exposure at Default*) | ¿Cuánto debe el cliente en el momento en que deja de pagar? |

**Ejemplo ilustrativo:** un crédito de $50,000 con PD de 5 % y LGD de 45 % tiene una pérdida esperada de 0.05 × 0.45 × 50,000 = **$1,125**. Es un costo "normal" del negocio: se cubre con la tasa de interés y con las reservas (en México, las *estimaciones preventivas para riesgos crediticios*). Las pérdidas mayores a lo esperado se cubren con capital.

### Punto de corte

Es el puntaje mínimo para aprobar. Se propone una estrategia de **tres zonas**, común en la industria:
- **Aprobación automática:** puntajes altos, riesgo bajo.
- **Revisión manual:** zona intermedia; un analista revisa información adicional (ingresos comprobables, garantías, historial con la institución).
- **Rechazo:** puntajes donde el cliente, en promedio, cuesta más de lo que deja.

### Supuestos (el dataset no trae montos ni recuperaciones)

*Give Me Some Credit* no incluye el monto del crédito, lo recuperado ni la tasa de interés. Por eso **LGD, EAD y margen son supuestos explícitos**, definidos en `src/perdida.py`, y al final se hace un **análisis de sensibilidad** para ver cuánto cambia la recomendación si cambian.

| Supuesto | Valor | Justificación |
|---|---|---|
| EAD por cliente | $50,000 | Crédito personal hipotético, igual para todos. Así la pérdida esperada es proporcional a la PD. |
| LGD | 45 % | Valor de referencia de Basilea para deuda sin garantía (enfoque básico). En crédito al consumo puede ser mayor; se evalúa en la sensibilidad. |
| Margen | 15 % de la EAD | Ingreso neto (intereses menos costo de fondeo y operación) de un cliente que paga, en el horizonte de 2 años. |

Todo se mide en el **mismo horizonte de 2 años** con que se definió a un cliente malo.

## Entorno

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "credit-risk-scorecard"
CARPETA_DRIVE = Path("/content/drive/MyDrive/credit-risk-scorecard")  # Donde está cs-training.csv

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q
    os.chdir(ruta_repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import config
from src.datos import copiar_datos_crudos
from src.limpieza import obtener_datos_limpios
from src.perdida import (
    EAD_POR_CLIENTE,
    LGD,
    MARGEN,
    asignar_banda,
    calibrar_pd_por_banda,
    comparar_escenarios,
    corte_por_utilidad,
    evaluar_zonas,
    graficar_perdida_por_banda,
    pd_equilibrio,
    puntaje_ks,
    sensibilidad,
    tabla_perdida_esperada,
)
from src.scorecard import construir_scorecard, pd_desde_puntaje

config.fijar_semilla()
config.crear_carpetas()
pd.set_option("display.float_format", "{:,.4f}".format)

In [ ]:
copiar_datos_crudos(CARPETA_DRIVE)
train, test = obtener_datos_limpios()
Y = config.COLUMNA_OBJETIVO

scorecard = construir_scorecard()
puntaje_train = scorecard["puntaje_train"]["puntaje"]
puntaje_test = scorecard["puntaje_test"]["puntaje"]

print(f"Supuestos: EAD = ${EAD_POR_CLIENTE:,} | LGD = {LGD:.0%} | Margen = {MARGEN:.0%}")

## 1. PD calibrada por banda de puntaje

En la etapa 4 vimos que el scorecard **ordena** muy bien a los clientes, pero su PD exacta exageraba el riesgo de los mejores clientes. Para calcular dinero se necesita una PD precisa, así que se **calibra**: la PD de cada banda de puntaje es la tasa de malos observada en **entrenamiento**. Después se comprueba en **prueba** que la tasa real se parece a la calibrada.

In [ ]:
calibracion = calibrar_pd_por_banda(puntaje_train, train[Y])

# Validación en prueba: tasa real por banda vs. PD calibrada en entrenamiento
validacion = (pd.DataFrame({"banda": asignar_banda(puntaje_test), "malo": test[Y]})
              .groupby("banda", observed=False)["malo"].agg(clientes_prueba="size", tasa_real_prueba="mean"))
validacion["pd_calibrada"] = calibracion["pd_calibrada"]
validacion["pd_original_scorecard"] = (pd.DataFrame({"banda": asignar_banda(puntaje_test),
                                                     "pd": pd_desde_puntaje(puntaje_test)})
                                       .groupby("banda", observed=False)["pd"].mean())

print("PD calibrada (entrenamiento)")
display(calibracion)
print("Validación en prueba")
display(validacion)

## 2. Pérdida esperada por banda

Se aplica la PD calibrada a los clientes de prueba, como si fueran solicitudes nuevas. Las columnas acumuladas se leen de arriba hacia abajo: *"si apruebo desde la mejor banda hasta esta, ¿qué % de clientes apruebo y cuánta pérdida y utilidad espero?"*.

In [ ]:
perdida = tabla_perdida_esperada(puntaje_test, calibracion["pd_calibrada"])

formato = {"pd": "{:.2%}", "pct_aprobacion_acum": "{:.2%}",
           **{c: "${:,.0f}" for c in ["exposicion", "perdida_esperada", "ingreso_esperado",
                                       "utilidad_esperada", "perdida_esperada_acum", "utilidad_esperada_acum"]}}
display(perdida.style.format(formato))

total_exposicion = perdida["exposicion"].sum()
total_perdida = perdida["perdida_esperada"].sum()
print(f"Cartera de prueba: exposición ${total_exposicion:,.0f} | pérdida esperada ${total_perdida:,.0f} "
      f"({total_perdida / total_exposicion:.2%} de la exposición)")

In [ ]:
graficar_perdida_por_banda(perdida)
plt.show()

## 3. Puntos de corte

**Corte de rechazo (rentabilidad).** Un cliente es rentable si lo que deja cuando paga supera lo que se pierde cuando no paga:

$$(1 - \text{PD}) \times \text{margen} > \text{PD} \times \text{LGD} \quad\Rightarrow\quad \text{PD} < \frac{\text{margen}}{\text{margen} + \text{LGD}}$$

Las bandas con PD mayor a ese límite tienen utilidad esperada negativa y se rechazan.

**Corte de aprobación automática (KS).** Se usa el puntaje donde el scorecard alcanza su KS en **entrenamiento**: el punto de máxima separación entre buenos y malos. Arriba de él se aprueba sin revisión; entre ambos cortes, revisión manual.

In [ ]:
limite_pd = pd_equilibrio()
corte_rechazo = corte_por_utilidad(calibracion["pd_calibrada"])
corte_aprobacion = puntaje_ks(puntaje_train, train[Y]) + 1   # Se aprueba por arriba del puntaje del KS

print(f"PD de equilibrio: {MARGEN:.0%} / ({MARGEN:.0%} + {LGD:.0%}) = {limite_pd:.2%}")
print(f"Corte de rechazo: puntaje < {corte_rechazo:.0f}")
print(f"Corte de aprobación automática (KS en entrenamiento): puntaje ≥ {corte_aprobacion:.0f}")

## 4. Resultado de la estrategia en prueba (con los resultados reales)

Ahora sí se usa lo que realmente pasó con los clientes de prueba: cuántos fueron malos en cada zona y cuánto se habría ganado o perdido.

In [ ]:
zonas = evaluar_zonas(puntaje_test, test[Y], corte_rechazo, corte_aprobacion)
formato_zonas = {"pct_clientes": "{:.2%}", "tasa_malos": "{:.2%}", "pct_malos_total": "{:.2%}",
                 "perdida": "${:,.0f}", "ingreso": "${:,.0f}", "utilidad": "${:,.0f}"}
zonas.style.format(formato_zonas)

In [ ]:
escenarios = comparar_escenarios(zonas)
escenarios.style.format({"pct_aprobados": "{:.2%}", "tasa_malos_aprobados": "{:.2%}",
                         "perdida": "${:,.0f}", "utilidad": "${:,.0f}"})

## 5. Sensibilidad a los supuestos

¿Qué pasa con el corte de rechazo si la LGD o el margen son distintos? Si la recomendación cambia mucho, los supuestos deben validarse con datos reales de la institución antes de implementarla.

In [ ]:
tabla_sensibilidad = sensibilidad(calibracion["pd_calibrada"], puntaje_test,
                                  lgds=[0.35, 0.45, 0.65, 0.85], margenes=[0.10, 0.15, 0.25])
tabla_sensibilidad.style.format({"lgd": "{:.0%}", "margen": "{:.0%}", "pd_equilibrio": "{:.2%}",
                                 "corte_rechazo": "{:.0f}", "pct_aprobacion": "{:.2%}"})

## Resumen de la etapa

- Se calibró la PD por banda de puntaje con entrenamiento y se validó en prueba.
- Se calculó la pérdida esperada (PD × LGD × EAD) por banda, con supuestos explícitos de LGD, EAD y margen.
- Se propuso una estrategia de tres zonas: rechazo por rentabilidad, revisión manual y aprobación automática desde el punto de máximo KS.
- Se evaluó la estrategia con los resultados reales de prueba y se analizó su sensibilidad a los supuestos.

**Siguiente paso:** README final con resumen de negocio y publicación del proyecto.